In [17]:
import numpy
import sys
import os
import matplotlib.pyplot as plt
# import skimage
import nrrd
from scipy import ndimage
import meshio
from skimage.morphology import skeletonize
from scipy.interpolate import splprep, splev
from sklearn.decomposition import PCA
from scipy.spatial import cKDTree
from scipy.interpolate import UnivariateSpline
import trimesh
from scipy.ndimage import median_filter

from scipy.spatial import cKDTree
from scipy.interpolate import UnivariateSpline
from skimage import measure
import tomllib

import pyvista as pv

from scipy.ndimage import uniform_filter1d

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import getSurfaceMesh, export_centerline,  tetra_mesh_from_stl, set_the_offset, tetra_shell_from_two_surfaces
from image_processing_functions import load_vti


with open("/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/config.toml", mode="rb") as f:
    config = tomllib.load(f)

sequence = "T1"

if sequence == "T2":
    base            = config["data"]["base"]
    input_file      = base + config["data"]["nii_file_T2"]
    output_folder   = config["data"]["output_folder_ext"]
    segment_filename = config["data"]["segment_filename_T2"]
    segment_folder  = config["data"]["segment_folder_T2"]
    centerline_folder = config["data"]["centerline_folder"]

else:
    base            = config["data"]["base"]
    input_file      = base + config["data"]["nii_file"]
    output_folder   = config["data"]["output_folder_ext"]
    segment_filename = config["data"]["segment_filename"]
    segment_folder  = config["data"]["segment_folder"]
    centerline_folder = config["data"]["centerline_folder"]

export_name = base + output_folder + segment_filename[:-8]


## Load image parameters

In [18]:
data, header = nrrd.read(base + segment_folder + segment_filename)

dirs = [numpy.array(d, dtype=float) for d in header["space directions"] if d is not None and not numpy.any(numpy.isnan(numpy.array(d, dtype=float)))]
dirs = numpy.array(dirs)

pixel_spacing = numpy.linalg.norm(dirs, axis=1)           # mm per voxel, in array axis order
direction = (dirs / pixel_spacing[:, None]).T          # columns = unit axis directions
origin = numpy.array(header["space origin"], dtype=float)

print("original pixel spacing = ", pixel_spacing)
print("segmentation origin = ", origin)

original pixel spacing =  [0.3125     0.3125     0.50000006]
segmentation origin =  [-55.95619965 -90.20069885  20.49090004]


In [ ]:
distance_A = -2.4
expansion_B = 2.0
distance_B = -10

# contact_axis_1 = 10*pixel_spacing[0]
# contact_axis_2 = 3.5*pixel_spacing[0]

contact_axis_1 = 8*pixel_spacing[0]
contact_axis_2 = 5*pixel_spacing[0]

h = 1.0


# distance_A = -2.2
# expansion_B = 2.4
# distance_B = -10

## Load segments


In [20]:

segments = []
i = 0
while f"Segment{i}_Name" in header:
    name  = header[f"Segment{i}_Name"]
    label = int(header.get(f"Segment{i}_LabelValue", i + 1))
    layer = int(header.get(f"Segment{i}_Layer", 0))
    vol   = data[layer] if data.ndim == 4 else data
    segments.append((name, vol == label))
    print(f"{name}: layer={layer}, value={label}, voxels={segments[-1][1].sum()}")
    i += 1

combined = numpy.zeros(segments[0][1].shape, dtype=numpy.uint8)
for new_id, (name, mask) in enumerate(segments, start=1):
    combined[mask] = new_id          # later segments overwrite earlier ones where they overlap

data = combined
id_max = int(data.max())
print(data.shape)


print()
print("original dimensions = ", data.shape)
print("max segment ID = ", id_max)
print()
print("Unique IDs")
with open(base + output_folder + "segment_IDs" + "_" + sequence + ".txt", "w") as f:
    for new_id, (name, _) in enumerate(segments, start=1):
        f.write(f"{name}\t{new_id}\n")
        print(f"{name}: new ID = {new_id}")


optical nerve: layer=0, value=1, voxels=15624
eyeball: layer=0, value=3, voxels=160561
lateral rectus: layer=0, value=4, voxels=14436
medial rectus: layer=0, value=5, voxels=13164
inferior rectus: layer=0, value=6, voxels=11961
superior rectus: layer=0, value=7, voxels=13568
lens: layer=0, value=2, voxels=1508
(145, 209, 101)

original dimensions =  (145, 209, 101)
max segment ID =  7

Unique IDs
optical nerve: new ID = 1
eyeball: new ID = 2
lateral rectus: new ID = 3
medial rectus: new ID = 4
inferior rectus: new ID = 5
superior rectus: new ID = 6
lens: new ID = 7


## Resample

In [21]:
pixel_spacing = numpy.asarray(pixel_spacing, dtype=float)
target = pixel_spacing.min()

if not numpy.allclose(pixel_spacing, target):
    factors = pixel_spacing / target
    data = ndimage.zoom(data, factors, order=0)
    print("resampling factors = ", factors)
    print("new dimensions = ", data.shape)

pixel_spacing = target

print("Final uniform pixel_spacing = ", pixel_spacing)

resampling factors =  [1.         1.         1.60000019]
new dimensions =  (145, 209, 162)
Final uniform pixel_spacing =  0.3124999999999999


## Define IDs

In [22]:
eyeball_id = 2
ignored_ids = [-1]
sclera_id = -1
lens_id = 7
ON_id = 1

muscle_ids_med_lat = [1,0]
muscle_ids_sup_inf = [3,2]
muscle_ids = [3,4,5,6]


smooth_ids = []

## Create initial meshes

In [23]:
for id in range(1,numpy.max(data)+1):
    if id not in ignored_ids:

        print("id = ", id)

        if len(numpy.where((data>id-0.5) & (data<id+0.5 ))[0])>0:

            mask = numpy.zeros(data.shape, dtype=numpy.uint8)
            mask[(data>id-0.5) & (data<id+0.5 )]=1

            if id == sclera_id:
                mask[(data>eyeball_id-0.5) & (data<eyeball_id+0.5 )]=1
                
            print(len(numpy.where(mask==1)[0]))

            if id in smooth_ids:
                mask = ndimage.median_filter(mask, size=2)
            mask[mask<0.5]=0
            mask[mask>=0.5]=1



            numpy.save(export_name +"_filtered_surf_id_"+str(id), mask)

            if id ==eyeball_id:
                mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing, True, "in" )

                mask_voxels = numpy.argwhere(mask == 1)  # shape (N, 3), coordinates in (z, y, x)
                # compute mean voxel position
                mesh_offset = mask_voxels.mean(axis=0)*pixel_spacing  
                numpy.savetxt(base + output_folder + "mesh_offset_"+sequence+".txt", mesh_offset)

            else:
                mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing, False )
            print()

id =  1
25127
Problem edges: 2
Open Hole Edges: 0
Non-Manifold Junctions: 2
 -> Remaining Problem edges: 46
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/mesh_extended/Segmentat_surf_id_1.stl

id =  2
258812
Problem edges: 0
Outer and inner surfaces written.

id =  3
23220
Problem edges: 12
Open Hole Edges: 0
Non-Manifold Junctions: 12
 -> Remaining Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/mesh_extended/Segmentat_surf_id_3.stl

id =  4
21124
Problem edges: 2
Open Hole Edges: 0
Non-Manifold Junctions: 2
 -> Remaining Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/mesh_extended/Segmentat_surf_id_4.stl

id =  5
19237
Problem edges: 9
Open Hole Edges: 0
Non-Manifold Junctions: 9
 -> Remaining Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/mesh_

## Define optical axis

In [24]:
mesh_lens = meshio.read(export_name +"_surf_id_"+str(lens_id)+".stl")
mesh_eyeball = meshio.read(export_name +"_surf_id_"+str(eyeball_id)+"_outer.stl")


lense_center = mesh_lens.points.mean(axis=0)

print("lense_center = ", lense_center)

X = mesh_lens.points - lense_center

_, _, vh = numpy.linalg.svd(X)

axis_direction_init = vh[-1]  # lens: first axis

def axis(t):
    return lense_center + t * axis_direction_init


axis_points_visual = [axis(-20.0), axis(20.0)]

export_centerline(axis_points_visual, 1.0, base + centerline_folder + "axial_length-init.vtp", [0,0,0])

export_centerline(axis_points_visual, 1.0, base + centerline_folder + "axial_length-centered-init.vtp", mesh_offset)
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

target_point_init =  axis(distance_A)/pixel_spacing

print("target_point = ", target_point_init)

/opt/homebrew/Caskroom/miniconda/base/envs/eye-project/lib/python3.13/site-packages/meshio/stl/_stl.py:40: RuntimeWarning: overflow encountered in scalar multiply
  if 84 + num_triangles * 50 == filesize_bytes:


lense_center =  [21.37719777  8.81420705 25.09814342]
target_point =  [68.7435123  35.87275818 80.59999513]


In [25]:
def fit_ellipsoid(points):
    """
    Least-squares fit of a general (arbitrarily rotated) ellipsoid.
    Returns center (3,), radii (3,), axes (3x3, one axis per column).
    """
    mean = points.mean(axis=0)
    scale = numpy.abs(points - mean).max()
    p = (points - mean) / scale                       # normalise for numerical stability
    x, y, z = p.T

    # Ax² + By² + Cz² + 2Dxy + 2Exz + 2Fyz + 2Gx + 2Hy + 2Iz = 1
    D_mat = numpy.c_[x*x, y*y, z*z, 2*x*y, 2*x*z, 2*y*z, 2*x, 2*y, 2*z]
    coef, *_ = numpy.linalg.lstsq(D_mat, numpy.ones(len(p)), rcond=None)
    A, B, C, D, E, F, G, H, I = coef

    Q = numpy.array([[A, D, E], [D, B, F], [E, F, C]])
    v = numpy.array([G, H, I])
    c = -numpy.linalg.solve(Q, v)                     # centre in normalised coordinates
    M = Q / (1.0 + c @ Q @ c)                         # (x-c)^T M (x-c) = 1

    eigvals, eigvecs = numpy.linalg.eigh(M)
    if numpy.any(eigvals <= 0):
        raise ValueError("Fitted quadric is not an ellipsoid")

    radii = scale / numpy.sqrt(eigvals)
    center = c * scale + mean
    return center, radii, eigvecs

def unit(v):
    return v / numpy.linalg.norm(v)

# Eyeball ellipsoid
eyeball_center, eyeball_radii, eyeball_axes = fit_ellipsoid(mesh_eyeball.points)
print("eyeball_center =", eyeball_center)
print("eyeball_radii  =", eyeball_radii)

# Pick the ellipsoid axis closest to the centre -> lens direction
to_lens = unit(lense_center - eyeball_center)
k = numpy.argmax(numpy.abs(eyeball_axes.T @ to_lens))
eyeball_dir = eyeball_axes[:, k]
if eyeball_dir @ to_lens < 0:
    eyeball_dir = -eyeball_dir                        # point towards the lens (anterior)
print(f"eyeball axis: radius {eyeball_radii[k]:.2f} mm, "
      f"{numpy.degrees(numpy.arccos(numpy.clip(eyeball_dir @ to_lens, -1, 1))):.1f} deg from centre->lens")

# Lens axis (least-variance direction), oriented the same way
lens_dir = unit(axis_direction_init)
if lens_dir @ eyeball_dir < 0:
    lens_dir = -lens_dir

angle = numpy.degrees(numpy.arccos(numpy.clip(lens_dir @ eyeball_dir, -1.0, 1.0)))
print(f"angle between lens and eyeball axis = {angle:.1f} deg")

MAX_ANGLE = 15.0
if angle <= MAX_ANGLE:
    axis_direction = unit(lens_dir + eyeball_dir)
    print("Using averaged axis")
else:
    axis_direction = lens_dir
    print("Eyeball axis too different, using lens axis only")


def axis(t):
    return lense_center + t * axis_direction

target_point =  axis(distance_A)/pixel_spacing


axis_points_visual = [axis(-20.0), axis(20.0)]

export_centerline(axis_points_visual, 1.0, base + centerline_folder + "axial_length.vtp", [0,0,0])

export_centerline(axis_points_visual, 1.0, base + centerline_folder + "axial_length-centered.vtp", mesh_offset)

eyeball_center = [21.50203176 16.74750835 26.7029466 ]
eyeball_radii  = [12.91101667 12.69277266 11.58412333]
eyeball axis: radius 11.58 mm, 16.4 deg from centre->lens
angle between lens and eyeball axis = 13.6 deg
Using averaged axis


## Section curve A

- plane perpendicular to optical axis
- through target_point 

In [26]:
eyeball_mesh = trimesh.load(export_name +"_surf_id_"+str(eyeball_id)+"_outer.stl")

section = eyeball_mesh.section(
    plane_origin=target_point*pixel_spacing,
    plane_normal=axis_direction
)

section_curve = []

for entity in section.entities:
    idx = entity.points  # ordered indices
    
    for i in range(len(idx) - 1):
        section_curve.append([idx[i], idx[i+1]])

meshio.write(
    base + centerline_folder + "section_curve_A.vtk",
    meshio.Mesh(
        points=section.vertices,
        cells=[("line", numpy.array(section_curve))]
    )
)

numpy.save(base + centerline_folder + "section_curve_A.npy", section.discrete[0])

## Section curve B (setting the distance from the eyeball)
- radial expansion


In [27]:
target_point_B =  axis(distance_B)/pixel_spacing

section_B = eyeball_mesh.section(
    plane_origin=target_point_B*pixel_spacing,
    plane_normal=axis_direction
)

section_curve_B = []

for entity in section_B.entities:
    idx = entity.points  # ordered indices
    
    for i in range(len(idx) - 1):
        section_curve_B.append([idx[i], idx[i+1]])


# Center point in physical space 
center_physical = target_point_B * pixel_spacing

original_vertices = numpy.array(section_B.vertices)

# Radial vectors from the center 
radial_vectors = original_vertices - center_physical

magnitudes = numpy.linalg.norm(radial_vectors, axis=1, keepdims=True)
normalized_vectors = radial_vectors / (magnitudes + 1e-8)



# Expand outward by xx mm 
expansion_distance = expansion_B 
expanded_vertices = original_vertices + (normalized_vectors * expansion_distance)

section_B.vertices = expanded_vertices

# 6. Rebuild the section_curve line segments using the new expanded coordinates
expanded_section_curve = []
for entity in section_B.entities:
    idx = entity.points  # ordered indices within this specific entity
    
    for i in range(len(idx) - 1):
        # We map the local entity indices directly to our expanded_vertices array
        expanded_section_curve.append([idx[i], idx[i+1]])

meshio.write(
    base + centerline_folder + "section_curve_B.vtk",
    meshio.Mesh(
        points=section_B.vertices,
        cells=[("line", numpy.array(expanded_section_curve))]
    )
)

## Functions: 
### Fit centerlines with/without additional constraints

In [28]:
def fit_target_constrained_spline(binned_centers, target_point, order, extension_factor = 0.05, smoothing_factor=3.5):
    """
    binned_centers: (N, 3) array of centers from the mask
    target_point: (3,) array [z, y, x] where the object should end
    """
    # 1. Append the target point to the end of your centers
    # We ensure target_point is shaped correctly for concatenation
    target_point = numpy.array(target_point).reshape(1, 3)
    constrained_points = numpy.concatenate([binned_centers, target_point], axis=0)
    
    # 2. Assign Weights (w)
    # We give the target point a very high weight (e.g., 10) 
    # and the noisy data points a lower weight (e.g., 1)
    weights = numpy.ones(len(constrained_points))
    weights[-1] = 20.0  # Force the spline to prioritize hitting the target
    
    # print("constrained_points = ", constrained_points)

    # 3. Fit the Spline
    # We use the transpose .T because splprep expects (3, N)
    try:
        # Note: 'w' is the weights, 's' is the overall smoothing
        tck, u = splprep(constrained_points.T, w=weights, s=smoothing_factor, k=order)
        
        # 4. Evaluate from 0 to 1
        # Since the target is now the last point in the data, 
        # u=1.0 will correspond to the target point.
        u_new = numpy.linspace(-extension_factor, 1.0, 160)
        smooth_centerline = numpy.array(splev(u_new, tck)).T
        
        return smooth_centerline
    except Exception as e:
        print(f"Constrained spline fit failed: {e}")
        return None
    

def fit_3_constrained_spline(binned_centers, target_1, target_2, target_3, order, extension_factor = 0.05, smoothing_factor=3.5):
    """
    Fits a spline by truncating binned centers at target_1 to avoid over-fitting,
    then sequentially appends target_2 and target_3.
    
    Ordering: [Body Part 1 (Up to Target 1)] -> [Target 1] -> [Target 2] -> [Target 3]
    """
    # 1. Standardize shapes
    target_1 = numpy.array(target_1).reshape(1, 3) # Intersects mid-body (Truncation point)
    target_2 = numpy.array(target_2).reshape(1, 3) # Intersects near end
    target_3 = numpy.array(target_3).reshape(1, 3) # Eye axis anchor (Absolute End)

    # 2. Find where target_1 belongs along the binned centers
    distances = numpy.linalg.norm(binned_centers - target_1, axis=1)
    split_idx = numpy.argmin(distances)

    # 3. Keep ONLY the binned centers that come strictly before target_1
    # We discard binned_centers[split_idx:] to prevent the spline from fight-fitting noise
    binned_truncated = binned_centers[:split_idx]

    # 4. Assemble the final sequence sequentially
    constrained_points = numpy.concatenate([
        binned_truncated,
        target_1,
        target_2,
        target_3
    ], axis=0)

    # 5. Assign precise weights based on the new truncated layout
    weights = numpy.ones(len(constrained_points))
    
    target_1_final_idx = len(binned_truncated)
    target_2_final_idx = len(constrained_points) - 2
    target_3_final_idx = len(constrained_points) - 1

    weights[target_1_final_idx] = 10.0  # Attract to target
    weights[target_2_final_idx] = 10.0  # Attract to target
    weights[target_3_final_idx] = 20.0  # Attract to target

    print(f"Truncated muscle body from {len(binned_centers)} down to {len(binned_truncated)} centers.")

    # 6. Fit the Spline
    try:
        tck, u = splprep(constrained_points.T, w=weights, s=smoothing_factor, k=order)
        
        u_new = numpy.linspace(-extension_factor, 1.0, 160)
        smooth_centerline = numpy.array(splev(u_new, tck)).T
        
        return smooth_centerline
    except Exception as e:
        print(f"Truncated 3-point constrained spline fit failed: {e}")
        return None


def fit_dual_constrained_spline(binned_centers, target_1, target_2, order, extension_factor = 0.05, smoothing_factor=3.5):
    """
    binned_centers: (N, 3) array of centers from the mask
    target_1: (3,) array where the object should begin
    target_2: (3,) array where the object should end
    """
    # 1. Prepare the sequence: [Start, Binned Data, End]
    points = [
        binned_centers,
        numpy.array(target_1).reshape(1, 3),
        numpy.array(target_2).reshape(1, 3)
    ]
    constrained_points = numpy.concatenate(points, axis=0)
    
    # 2. Assign Weights
    # Weights[0] is Start, Weights[-1] is End. 
    # High values force the spline to hit those exact coordinates.
    weights = numpy.ones(len(constrained_points))
    weights[0] = 50.0  # Force start
    weights[-1] = 50.0 # Force end
    
    # 3. Fit the Spline
    # We pass the weights array to 'w'
    try:
        tck, u = splprep(constrained_points.T, w=weights, s=smoothing_factor, k=order)
        
        # 4. Evaluate
        # u=0.0 will be target_start, u=1.0 will be target_end
        u_new = numpy.linspace(-extension_factor, 1.0, 160)
        smooth_centerline = numpy.array(splev(u_new, tck)).T
        
        return smooth_centerline
    except Exception as e:
        print(f"Dual constrained spline fit failed: {e}")
        return None

def fit_analytical_centerline(mask, target_point, ignored_nodes, order, n_bins=20, extension_factor=0.2, both_sides = False):
        # 1. Get all voxel coordinates (Z, Y, X)
        points = numpy.argwhere(mask == 1).astype(float)
        
        if len(points) < 10:
            return None

        # 2. Find the main orientation using PCA
        pca = PCA(n_components=1)
        # Project points onto the first principal component 
        projection = pca.fit_transform(points)


        # =========================================================================
        # 2.5 FORCE ORIENTATION TOWARD EYE 
        # =========================================================================
        if len(target_point) > 0:
            # The direction of the PCA axis vector in 3D space
            pca_axis = pca.components_[0] 
            
            # Vector pointing from the muscle center of mass to the target point
            center_of_mass = points.mean(axis=0)
            direction_to_target = numpy.array(target_point[-1]) - center_of_mass
            
            # If the dot product is negative, the PCA axis points AWAY from the target.
            if numpy.dot(pca_axis, direction_to_target) < 0:
                print("reverse")
                projection = -projection

        
        # 3. Sort points based on their position along the principal axis
        sort_idx = numpy.argsort(projection.flatten())

        sort_idx = sort_idx[int(len(sort_idx)*ignored_nodes):int(len(sort_idx)*(1-ignored_nodes))]

        sorted_points = points[sort_idx]
        sorted_projection = projection[sort_idx].flatten()

        # 4. Create "Binned Centers" to handle the noise
        # Fit to the average center of 'n' slices
        bins = numpy.linspace(sorted_projection.min(), sorted_projection.max(), n_bins)
        binned_centers = []
        
        for i in range(len(bins)-1):
            # Find points that fall within this slice of the object
            mask_slice = (sorted_projection >= bins[i]) & (sorted_projection < bins[i+1])
            if numpy.any(mask_slice):
                center = sorted_points[mask_slice].mean(axis=0)
                binned_centers.append(center)
        
        binned_centers = numpy.array(binned_centers)

        # # 5. Fit the Spline to the centers
        tck, u = splprep(binned_centers.T, s=3.5, k=2)
        
        # # 6. Extend the object
        if both_sides:
            smooth_centerline = numpy.array(splev(numpy.linspace(-extension_factor, 1.0 + extension_factor , 100), tck)).T
        else:
            # # Evaluating from 0.0 to 1+extension extends the centerline
            smooth_centerline = numpy.array(splev(numpy.linspace(-extension_factor, 1.0 , 100), tck)).T

        if len(target_point)==0:
            return smooth_centerline
        
        if len(target_point)==1:
            smooth_centerline_extended = fit_target_constrained_spline(binned_centers, target_point[0], order, extension_factor)
            return smooth_centerline, smooth_centerline_extended
        
        if len(target_point)==2:
            smooth_centerline_extended = fit_dual_constrained_spline(binned_centers, target_point[0], target_point[1], order, extension_factor )
            return smooth_centerline, smooth_centerline_extended

        if len(target_point)==3:
            smooth_centerline_extended = fit_3_constrained_spline(binned_centers, target_point[0], target_point[1], target_point[2], order, extension_factor )
            return smooth_centerline, smooth_centerline_extended    


## Functions:
### Get insertion points ~ section curve / fitted planes

In [29]:
def fit_plane_to_two_centerlines(centerline1, centerline2):
    # 1. Combine all points from both centerlines into one cloud
    points = numpy.vstack((centerline1, centerline2))
    
    # 2. Calculate the centroid (the average point)
    centroid = numpy.mean(points, axis=0)
    
    # 3. Center the points by subtracting the centroid
    centered_points = points - centroid
    
    # 4. Use Singular Value Decomposition
    _, _, vh = numpy.linalg.svd(centered_points)
    
    # 5. The normal vector 'n' is the last row of Vh 
    # (the direction with the least variance/spread)
    normal = vh[-1]
    
    return centroid, normal/numpy.linalg.norm(normal)

def find_plane_curve_intersections(section, section_curve, plane_centroid, plane_normal):
    intersections = []
    
    # Normalize plane normal just in case
    plane_normal = plane_normal / numpy.linalg.norm(plane_normal)
    
    vertices = section.vertices
    
    for start_idx, end_idx in section_curve:
        p1 = vertices[start_idx]
        p2 = vertices[end_idx]
        
        # Signed distances to the plane
        d1 = numpy.dot(p1 - plane_centroid, plane_normal)
        d2 = numpy.dot(p2 - plane_centroid, plane_normal)
        
        # Check if the segment crosses the plane
        if d1 * d2 < 0:
            # Linear interpolation to find the exact crossing point
            t = abs(d1) / (abs(d1) + abs(d2))
            intersect_pt = p1 + t * (p2 - p1)
            intersections.append(intersect_pt)
            
    return numpy.array(intersections)



## Compute original centerlines & Define planes and target points

In [30]:
extended_centerlines = []
centerlines = []
extended_centerlines_1 = []

order = 3
# target_point = point on the lense axis

for muscle_id in muscle_ids:

    mask = numpy.zeros(data.shape, dtype=numpy.uint8)
    mask[(data>muscle_id-0.5) & (data<muscle_id+0.5 )]=1

    ignored_nodes = 0.01

    centerline, centerline_ext_1 = fit_analytical_centerline(mask, [target_point], ignored_nodes, order, n_bins=10, extension_factor=0.15)
    
    centerlines.append(centerline)
    extended_centerlines.append(centerline_ext_1)

    export_centerline(centerline_ext_1, pixel_spacing, base + centerline_folder + "centerline_exttended_1_id_"+str(muscle_id)+".vtp", [0,0,0])
    export_centerline(centerline, pixel_spacing, base + centerline_folder + "centerline_init_id_"+str(muscle_id)+".vtp", [0,0,0])

    numpy.save(base + centerline_folder + "centerline_init_id_"+str(muscle_id)+".npy", centerline)

print("-------------------------")

point_hor, normal_vector_hor = fit_plane_to_two_centerlines(extended_centerlines[muscle_ids_med_lat[0]], extended_centerlines[muscle_ids_med_lat[1]])
point_vert, normal_vector_vert = fit_plane_to_two_centerlines(extended_centerlines[muscle_ids_sup_inf[0]], extended_centerlines[muscle_ids_sup_inf[1]])

print("Check of angle  = ", numpy.dot(normal_vector_hor, normal_vector_vert))

# Plane 1 intersections
pts_plane_horizontal = find_plane_curve_intersections(section, section_curve, point_hor*pixel_spacing, normal_vector_hor)
# Plane 2 intersections
pts_plane_vertical = find_plane_curve_intersections(section, section_curve, point_vert*pixel_spacing, normal_vector_vert)


# Plane 1B intersections
pts_plane_B_horizontal = find_plane_curve_intersections(section_B, section_curve_B, point_hor*pixel_spacing, normal_vector_hor)
# Plane 2B intersections
pts_plane_B_vertical = find_plane_curve_intersections(section_B, section_curve_B, point_vert*pixel_spacing, normal_vector_vert)



reverse
reverse
reverse
reverse
-------------------------
Check of angle  =  0.11740433592103146


## ON centerline

In [31]:
mask = numpy.zeros(data.shape, dtype=numpy.uint8)
mask[(data>ON_id-0.5) & (data<ON_id+0.5 )]=1

ignored_nodes = 0.0

centerline, centerline_ext_1 = fit_analytical_centerline(mask, [target_point], ignored_nodes, order, n_bins=10, extension_factor=0.1)

centerlines.append(centerline)
extended_centerlines.append(centerline_ext_1)

export_centerline(centerline_ext_1, pixel_spacing, base + centerline_folder + "centerline_exttended_1_id_"+str(ON_id)+".vtp", [0,0,0])
export_centerline(centerline, pixel_spacing, base + centerline_folder + "centerline_init_id_"+str(ON_id)+".vtp", [0,0,0])

numpy.save(base + centerline_folder + "centerline_init_id_"+str(ON_id)+".npy", centerline)

reverse


## Extend centerlines

In [32]:
for id in range(len(muscle_ids)):

    print(id)

    if id in muscle_ids_med_lat:
        print("muslce id ", muscle_ids[id],    "horizontal")

        targets_1 = pts_plane_horizontal/pixel_spacing
        targets_1_B = pts_plane_B_horizontal/pixel_spacing

    if id in muscle_ids_sup_inf:
        print("     muslce id ", muscle_ids[id],   " vertical")

        targets_1 = pts_plane_vertical/pixel_spacing
        targets_1_B = pts_plane_B_vertical/pixel_spacing

    # Extract coordinates of the current muscle mask
    mask = numpy.zeros(data.shape, dtype=numpy.uint8)
    mask[(data>muscle_ids[id]-0.5) & (data<muscle_ids[id]+0.5 )]=1
        
    muscle_voxels = numpy.argwhere(mask) * pixel_spacing
    tree = cKDTree(muscle_voxels)
    
    # Query distance to the chosen target points
    dist_to_target_1, _ = tree.query(targets_1 * pixel_spacing)
    dist_to_target_1_B, _ = tree.query(targets_1_B * pixel_spacing)
    
    print("        -> Distance to target_1: ", dist_to_target_1)
    print("        -> Distance to target_1_B: ", dist_to_target_1_B)

    target_1 = targets_1[numpy.argmin(dist_to_target_1)]
    target_1_B = targets_1_B[numpy.argmin(dist_to_target_1_B)]


    _, centerline_ext_2 = fit_analytical_centerline(mask, [target_1_B, target_1, target_point], ignored_nodes, order, n_bins=10, extension_factor=0.1)

    extended_centerlines_1.append(centerline_ext_2)


    export_centerline(centerline_ext_2, pixel_spacing, base + centerline_folder + "centerline_exttended_2_id_"+str(muscle_ids[id])+".vtp", [0,0,0])
    export_centerline(centerline_ext_2, pixel_spacing, base + centerline_folder + "centerline_exttended_2_id_"+str(muscle_ids[id])+"-centered.vtp", mesh_offset)
    numpy.save(base + centerline_folder + "centerline_exttended_2_id_"+str(muscle_ids[id])+".npy", centerline_ext_2)
    
    print("-------------------------")


0
muslce id  3 horizontal
        -> Distance to target_1:  [ 8.19490097 24.92327962]
        -> Distance to target_1_B:  [ 0.80581863 24.03225883]
reverse
Truncated muscle body from 9 down to 8 centers.
-------------------------
1
muslce id  4 horizontal
        -> Distance to target_1:  [24.27299434  3.98848559]
        -> Distance to target_1_B:  [26.64462371  0.20003309]
reverse
Truncated muscle body from 9 down to 7 centers.
-------------------------
2
     muslce id  5  vertical
        -> Distance to target_1:  [23.94833212  3.58031639]
        -> Distance to target_1_B:  [26.20802001  0.18837975]
reverse
Truncated muscle body from 9 down to 7 centers.
-------------------------
3
     muslce id  6  vertical
        -> Distance to target_1:  [ 7.70313905 24.69867224]
        -> Distance to target_1_B:  [ 0.29101896 23.71735502]
reverse
Truncated muscle body from 9 down to 8 centers.
-------------------------
